In [27]:
import numpy as np
from scipy.optimize import root

In [28]:
s = {
    'ORL' : 0, 'ORR' : 1, 'CRL' : 2, 'CRR' : 3, 'LRL' : 4, 'LRR' : 5, 'RRL' : 6, 'RRR' : 7
}

In [29]:
o = {
    'OCL' : 0, 'OCR' : 1, 'ORW' : 2, 'ONR' : 3, 'CCL' : 4, 'CCR' : 5, 'CRW' : 6, 'CNR' : 7,
    'LCL' : 8, 'LCR' : 9, 'LRW' : 10, 'LNR' : 11, 'RCL' : 12, 'RCR' : 13, 'RRW' : 14, 'RNR' : 15
}

In [30]:
u = {
    'O' : 0, 'C' : 1, 'L' : 2, 'R' : 3
}

In [31]:
# p(s_0)
p_D = np.zeros(8)
p_D[s['ORL']] = 0.5
p_D[s['ORR']] = 0.5

In [32]:
# p(o|s)
p_A = np.zeros((16, 8))
alpha = 0.9
p_A[o['OCL'], s['ORL']] = 0.5
p_A[o['OCL'], s['ORR']] = 0.5
p_A[o['LRW'], s['LRL']] = alpha
p_A[o['LRW'], s['LRR']] = 1 - alpha
p_A[o['RRW'], s['RRL']] = 1 - alpha
p_A[o['RRW'], s['RRR']] = alpha
p_A[o['CCL'], s['CRL']] = 1
p_A[o['CCR'], s['CRR']] = 1
p_A[o['OCR'], s['ORL']] = 0.5
p_A[o['OCR'], s['ORR']] = 0.5
p_A[o['LNR'], s['LRL']] = 1 - alpha
p_A[o['LNR'], s['LRR']] = alpha
p_A[o['RNR'], s['RRL']] = alpha
p_A[o['RNR'], s['RRR']] = 1 - alpha

In [33]:
# p(u)
p_U = np.ones(4) * 0.25

In [34]:
# p(s_k|s_k-1,u_k)
p_B = np.zeros((8, 8, 4))
p_B[s['ORL'], s['ORL'], u['O']] = 1
p_B[s['ORL'], s['CRL'], u['O']] = 1
p_B[s['ORL'], s['RRL'], u['O']] = 1
p_B[s['ORL'], s['LRL'], u['O']] = 1
p_B[s['CRL'], s['ORL'], u['C']] = 1
p_B[s['CRL'], s['CRL'], u['C']] = 1
p_B[s['ORL'], s['RRL'], u['C']] = 1
p_B[s['ORL'], s['LRL'], u['C']] = 1
p_B[s['LRL'], s['ORL'], u['L']] = 1
p_B[s['LRL'], s['CRL'], u['L']] = 1
p_B[s['ORL'], s['RRL'], u['L']] = 1
p_B[s['ORL'], s['LRL'], u['L']] = 1
p_B[s['RRL'], s['ORL'], u['R']] = 1
p_B[s['RRL'], s['CRL'], u['R']] = 1
p_B[s['ORL'], s['RRL'], u['R']] = 1
p_B[s['ORL'], s['LRL'], u['R']] = 1
p_B[s['ORR'], s['ORR'], u['O']] = 1
p_B[s['ORR'], s['CRR'], u['O']] = 1
p_B[s['ORR'], s['RRR'], u['O']] = 1
p_B[s['ORR'], s['LRR'], u['O']] = 1
p_B[s['CRR'], s['ORR'], u['C']] = 1
p_B[s['CRR'], s['CRR'], u['C']] = 1
p_B[s['ORR'], s['RRR'], u['C']] = 1
p_B[s['ORR'], s['LRR'], u['C']] = 1
p_B[s['LRR'], s['ORR'], u['L']] = 1
p_B[s['LRR'], s['CRR'], u['L']] = 1
p_B[s['ORR'], s['RRR'], u['L']] = 1
p_B[s['ORR'], s['LRR'], u['L']] = 1
p_B[s['RRR'], s['ORR'], u['R']] = 1
p_B[s['RRR'], s['CRR'], u['R']] = 1
p_B[s['ORR'], s['RRR'], u['R']] = 1
p_B[s['ORR'], s['LRR'], u['R']] = 1

In [35]:
# p~(o)
p_C = np.ones(16)
c = 2
p_C = p_C * (1 / (4 * np.exp(c) + 4 * np.exp(-c) + 8))
p_C[o['ORW']] = np.exp(c) / (4 * np.exp(c) + 4 * np.exp(-c) + 8)
p_C[o['CRW']] = np.exp(c) / (4 * np.exp(c) + 4 * np.exp(-c) + 8)
p_C[o['LRW']] = np.exp(c) / (4 * np.exp(c) + 4 * np.exp(-c) + 8)
p_C[o['RRW']] = np.exp(c) / (4 * np.exp(c) + 4 * np.exp(-c) + 8)
p_C[o['ONR']] = np.exp(-c) / (4 * np.exp(c) + 4 * np.exp(-c) + 8)
p_C[o['CNR']] = np.exp(-c) / (4 * np.exp(c) + 4 * np.exp(-c) + 8)
p_C[o['LNR']] = np.exp(-c) / (4 * np.exp(c) + 4 * np.exp(-c) + 8)
p_C[o['RNR']] = np.exp(-c) / (4 * np.exp(c) + 4 * np.exp(-c) + 8)

In [ ]:
def optimize_q_s(p_A, p_C, *messages):
    """
    Berechnet die optimale variationelle Verteilung q(s) über das Newton-Verfahren.
    
    Inputs:
    - p_A:      numpy array (16, 8) - Beobachtungsmatrix p(o | s)
    - p_C:      numpy array (16,)   - Ziel-Prior p~(o)
    - right_s: numpy array (8,)    - Eingehende Nachricht mu_rightarrow
    - left_s:  numpy array (8,)    - Eingehende Nachricht mu_leftarrow
    
    Returns:
    - q_s_star: numpy array (8,)   - Die konvergierte, optimale Verteilung
    """
    
    # --- 1. Verschachtelte Hilfsfunktionen ---
    
    def softmax(x):
        """Numerisch stabiler Softmax"""
        e_x = np.exp(x - np.max(x))
        return e_x / e_x.sum()

    def error_function(q_s):
        """Berechnet das Residuum F(q_s) für den Solver"""
        epsilon = 1e-12
        
        # Vorhersage der Beobachtung (q_o)
        q_o = p_A @ q_s
        
        # Spalten-Entropie von p_A berechnen (h_p_A)
        h_p_A = -np.sum(p_A * np.log(p_A + epsilon), axis=0)
        
        # Fehler-Term rho berechnen
        rho = p_A.T @ (np.log(p_C + epsilon) - np.log(q_o + epsilon)) - h_p_A
        
        # Eingehende Nachrichten im Log-Raum addieren
        #log_messages = np.log(right_s + epsilon) + np.log(left_s + epsilon)
        log_messages = sum(np.log(msg) for msg in messages)
        
        # Rückgabe der Differenz (soll 0 werden)
        return q_s - softmax(rho + log_messages)

    # --- 2. Vorbereitung und Optimierung ---
    
    # Bestimme die Anzahl der Zustände dynamisch (hier: 8)
    num_states = p_A.shape[1]
    
    # Startwert für den Solver (Gleichverteilung)
    q_s_init = np.ones(num_states) / num_states
    
    # Solver aufrufen (die verschachtelte Funktion greift automatisch auf p_A, etc. zu)
    solution = root(error_function, x0=q_s_init, method='hybr')
    
    # --- 3. Ergebnisverarbeitung ---
    
    if solution.success:
        # Ergebnis auslesen
        q_s_star = solution.x
        
        # Numerische Sicherheitsmaßnahme: Werte zwischen 0 und 1 clippen und normalisieren
        q_s_star = np.clip(q_s_star, 1e-12, 1.0)
        q_s_star /= q_s_star.sum()
        
        return q_s_star
    else:
        # Falls der Solver mal nicht konvergiert, werfen wir einen sauberen Fehler
        raise RuntimeError(f"Newton-Solver hat keine Lösung gefunden: {solution.message}")

In [37]:
def optimize_q_s2(p_A, p_C, right_s2):
    """
    Berechnet die optimale variationelle Verteilung q(s_2) über das Newton-Verfahren.
    
    Inputs:
    - p_A:      numpy array (16, 8) - Beobachtungsmatrix p(o_2 | s_2)
    - p_C:      numpy array (16,)   - Ziel-Prior p~(o_2)
    - right_s2: numpy array (8,)    - Einzige eingehende Nachricht mu_rightarrow
    
    Returns:
    - q_s2_star: numpy array (8,)   - Die konvergierte, optimale Verteilung
    """
    
    def softmax(x):
        """Numerisch stabiler Softmax"""
        e_x = np.exp(x - np.max(x))
        return e_x / e_x.sum()

    def error_function(q_s2):
        """Berechnet das Residuum F(q_s2) für den Solver"""
        epsilon = 1e-12
        
        # Vorhersage der Beobachtung (q_o2)
        q_o2 = p_A @ q_s2
        
        # Spalten-Entropie von p_A berechnen (h_p_A)
        h_p_A = -np.sum(p_A * np.log(p_A + epsilon), axis=0)
        
        # Fehler-Term rho berechnen
        rho = p_A.T @ (np.log(p_C + epsilon) - np.log(q_o2 + epsilon)) - h_p_A
        
        # Eingehende Nachricht im Log-Raum (hier gibt es nur noch right_s2!)
        log_messages = np.log(right_s2 + epsilon)
        
        # Rückgabe der Differenz (soll 0 werden)
        return q_s2 - softmax(rho + log_messages)

    # Bestimme die Anzahl der Zustände dynamisch
    num_states = p_A.shape[1]
    
    # Startwert für den Solver (Gleichverteilung)
    q_s2_init = np.ones(num_states) / num_states
    
    # Solver aufrufen
    solution = root(error_function, x0=q_s2_init, method='hybr')
    
    if solution.success:
        # Ergebnis auslesen
        q_s2_star = solution.x
        
        # Numerische Sicherheitsmaßnahme: Werte zwischen 0 und 1 clippen und normalisieren
        q_s2_star = np.clip(q_s2_star, 1e-12, 1.0)
        q_s2_star /= q_s2_star.sum()
        
        return q_s2_star
    else:
        # Falls der Solver mal nicht konvergiert
        raise RuntimeError(f"Newton-Solver hat keine Lösung gefunden: {solution.message}")

In [55]:
q_s1 = np.ones(8) * (1/8)
q_s2 = np.ones(8) * (1/8)
q_o1 = np.ones(16) * (1/16)
q_o2 = np.ones(16) * (1/16)
eps = np.finfo(float).eps

In [65]:
right_s0 = p_D
down_u1 = p_U
right_s1 = np.einsum('ijk,j,k->i', p_B, right_s0, down_u1)

up_s1 = np.exp(np.einsum('ij,ij->j', p_A, np.log(np.clip((p_A * p_C[:, np.newaxis]) / np.clip(q_o1[:, np.newaxis], a_min=eps, a_max=None), a_min=eps, a_max=None))))
down_u2 = p_U
right_s2 = np.einsum('ijk,j,j,k->i', p_B, right_s1, up_s1, down_u2)

up_s2 = np.exp(np.einsum('ij,ij->j', p_A, np.log(np.clip((p_A * p_C[:, np.newaxis]) / np.clip(q_o2[:, np.newaxis], a_min=eps, a_max=None), a_min=eps, a_max=None))))
up_u2 = np.einsum('ijk,i,j,j->k', p_B, up_s2, right_s1, up_s1)
left_s1 = np.einsum('ijk,k,i->j', p_B, down_u2, up_s2)
up_u1 = np.einsum('ijk,i,i,j->k', p_B, left_s1, up_s1, right_s0)
left_s0 = np.einsum('ijk,k,i,i->j', p_B, down_u1, left_s1, up_s1)

q_s0 = (right_s0 * left_s0) / np.sum(right_s0 * left_s0)
#q_s1 = (right_s1 * up_s1 * left_s1) / np.sum(right_s1 * up_s1 * left_s1)
q_s1 = optimize_q_s(p_A, p_C, right_s1, left_s1)
#q_s2 = (right_s2 * up_s2) / np.sum(right_s2 * up_s2)
q_s2 = optimize_q_s(p_A, p_C, right_s2)
q_o1 = np.einsum('i,ji->j', q_s1, p_A)
q_o2 = np.einsum('i,ji->j', q_s2, p_A)
q_u1 = (down_u1 * up_u1) / np.sum(down_u1 * up_u1)
q_u2 = (down_u2 * up_u2) / np.sum(down_u2 * up_u2)

print('right_s0: ' + str(np.round(right_s0, 3)))
print('down_u0: ' + str(np.round(down_u1, 3)))
print('right_s1: ' + str(np.round(right_s1, 3)))
print('up_s1: ' + str(np.round(up_s1, 3)))
print('down_u1: ' + str(np.round(down_u2, 3)))
print('right_s2: ' + str(np.round(right_s2, 3)))
print('up_s2: ' + str(np.round(up_s2, 3)))
print('up_u1: ' + str(np.round(up_u2, 3)))
print('left_s1: ' + str(np.round(left_s1, 3)))
print('up_u0: ' + str(np.round(up_u1, 3)))
print('left_s0: ' + str(np.round(left_s0, 3)))
print()
print('q(s_0) = ' + str(np.round(q_s0, 3)))
print('q(s_1) = ' + str(np.round(q_s1, 3)))
print('q(s_2) = ' + str(np.round(q_s2, 3)))
print('q(o_1) = ' + str(np.round(q_o1, 3)))
print('q(o_2) = ' + str(np.round(q_o2, 3)))
print('q(u_1) = ' + str(np.round(q_u1, 3)))
print('q(u_2) = ' + str(np.round(q_u2, 3)))

right_s0: [0.5 0.5 0.  0.  0.  0.  0.  0. ]
down_u0: [0.25 0.25 0.25 0.25]
right_s1: [0.125 0.125 0.125 0.125 0.125 0.125 0.125 0.125]
up_s1: [0.105 0.105 0.148 0.148 0.677 0.087 0.087 0.677]
down_u1: [0.25 0.25 0.25 0.25]
right_s2: [0.103 0.103 0.008 0.008 0.008 0.008 0.008 0.008]
up_s2: [0.061 0.061 0.313 0.313 0.666 0.086 0.086 0.666]
up_u1: [0.016 0.032 0.035 0.035]
left_s1: [0.281 0.281 0.281 0.281 0.061 0.061 0.061 0.061]
up_u0: [0.03  0.042 0.023 0.023]
left_s0: [0.03 0.03 0.03 0.03 0.03 0.03 0.03 0.03]

q(s_0) = [0.5 0.5 0.  0.  0.  0.  0.  0. ]
q(s_1) = [0.125 0.125 0.177 0.177 0.175 0.023 0.023 0.175]
q(s_2) = [0.214 0.214 0.084 0.084 0.179 0.023 0.023 0.179]
q(o_1) = [0.125 0.125 0.    0.    0.177 0.177 0.    0.    0.    0.    0.16  0.038
 0.    0.    0.16  0.038]
q(o_2) = [0.214 0.214 0.    0.    0.084 0.084 0.    0.    0.    0.    0.163 0.039
 0.    0.    0.163 0.039]
q(u_1) = [0.25  0.354 0.198 0.198]
q(u_2) = [0.132 0.267 0.301 0.301]


In [29]:
right_s0 = p_D
down_u1 = np.array([0, 1, 0, 0])
#right_s1 = np.einsum('ij,j->i', p_B[:, :, u['C']], right_s0)
right_s1 = np.einsum('ijk,j,k->i', p_B, right_s0, down_u1)
q_o1 = np.einsum('i,ji->j', q_s1, p_A)
#up_s1 = np.exp(np.einsum('ij,ij->j', p_A, np.log(np.clip((p_A * p_C[:, np.newaxis]) / np.clip(q_o1[:, np.newaxis], a_min=eps, a_max=None), a_min=eps, a_max=None))))
up_s1 = p_A[o['CCL']]
down_u2 = p_U
right_s2 = np.einsum('ijk,j,j,k->i', p_B, right_s1, up_s1, down_u2)
q_o2 = np.einsum('i,ji->j', q_s2, p_A)
up_s2 = np.exp(np.einsum('ij,ij->j', p_A, np.log(np.clip((p_A * p_C[:, np.newaxis]) / np.clip(q_o2[:, np.newaxis], a_min=eps, a_max=None), a_min=eps, a_max=None))))
up_u2 = np.einsum('ijk,i,j,j->k', p_B, up_s2, right_s1, up_s1)
left_s1 = np.einsum('ijk,k,i->j', p_B, down_u2, up_s2)
#up_u1 = np.einsum('ijk,i,i,j->k', p_B, left_s1, up_s1, right_s0)
left_s0 = np.einsum('ij,i,i->j', p_B[:, :, u['C']], left_s1, up_s1)

q_s0 = (right_s0 * left_s0) / np.sum(right_s0 * left_s0)
q_s1 = (right_s1 * up_s1 * left_s1) / np.sum(right_s1 * up_s1 * left_s1)
#q_s1 = optimize_q_s1(p_A, p_C, right_s1, left_s1)
q_s2 = (right_s2 * up_s2) / np.sum(right_s2 * up_s2)
#q_s2 = optimize_q_s2(p_A, p_C, right_s2)
q_u1 = (down_u1 * up_u1) / np.sum(down_u1 * up_u1)
q_u2 = (down_u2 * up_u2) / np.sum(down_u2 * up_u2)

print('q(s_0) = ' + str(np.round(q_s0, 3)))
print('q(s_1) = ' + str(np.round(q_s1, 3)))
print('q(s_2) = ' + str(np.round(q_s2, 3)))
print('q(o_1) = ' + str(np.round(q_o1, 3)))
print('q(o_2) = ' + str(np.round(q_o2, 3)))
print('q(u_1) = ' + str(np.round(q_u1, 3)))
print('q(u_2) = ' + str(np.round(q_u2, 3)))

q(s_0) = [1. 0. 0. 0. 0. 0. 0. 0.]
q(s_1) = [0. 0. 1. 0. 0. 0. 0. 0.]
q(s_2) = [0.059 0.    0.277 0.    0.588 0.    0.076 0.   ]
q(o_1) = [0.111 0.111 0.    0.    0.157 0.157 0.    0.    0.    0.    0.188 0.045
 0.    0.    0.188 0.045]
q(o_2) = [0.204 0.204 0.    0.    0.087 0.087 0.    0.    0.    0.    0.169 0.04
 0.    0.    0.169 0.04 ]
q(u_1) = [0. 1. 0. 0.]
q(u_2) = [0.059 0.277 0.588 0.076]
